## 1 Introduction
Bài nghiên cứu này tập trung vào việc làm rõ tầm ảnh hưởng của các đặc tính âm thanh kỹ thuật số đối với mức độ thành công (thương mại) của một tác phẩm âm nhạc.

*   **Câu hỏi nghiên cứu:** Các chỉ số âm nhạc như độ nhảy (`danceability_%`) hay năng lượng (`energy_%`) có tác động thuận chiều đến tổng lượt stream không?
*   **Giả thuyết nghiên cứu:**
    *   $H_0$: Độ nhảy và năng lượng không có tác động hoặc tác động nghịch chiều đến lượt stream.
    *   $H_1$: Độ nhảy và năng lượng có tác động thuận chiều tích cực đến lượt stream.

## 2 Mô tả dữ liệu và tiền xử lý dữ liệu

### a. Mô tả dữ liệu
#### Nguồn dữ liệu

Dataset: Top Spotify Songs 2023 Dataset (Kaggle)

Tập dữ liệu bao gồm các thông tin về những bài hát thành công nhất trên nền tảng Spotify trong năm 2023. Để phục vụ cho câu hỏi nghiên cứu về mối quan hệ giữa đặc tính âm thanh và mức độ thành công thương mại, các biến số được lựa chọn và phân loại cấu trúc như sau:

| Tên biến | Kiểu dữ liệu | Vai trò trong mô hình | Ý nghĩa chức năng |
| :--- | :--- | :--- | :--- |
| `streams` | Số nguyên (`int64`) | **Biến phụ thuộc** (Target - $y$) | Tổng số lượt nghe tích lũy trên Spotify. Đây là thước đo chính cho sự thành công thương mại. |
| `log_streams` | Số thực (`float64`) | **Biến phụ thuộc** (Sau biến đổi) | Giá trị Logarit tự nhiên của lượt stream ($\ln(1 + \text{streams})$), dùng để xử lý phân phối lệch và đảm bảo giả định hồi quy. |
| `danceability_%` | Số nguyên (`int64`) | **Biến độc lập** (Predictor - $x_1$) | Độ phù hợp của bài hát đối với việc nhảy nhảy (thang điểm 0 - 100%), dựa trên nhịp điệu, độ ổn định của nhịp. |
| `energy_%` | Số nguyên (`int64`) | **Biến độc lập** (Predictor - $x_2$) | Phép đo tốc độ và cường độ âm thanh (thang điểm 0 - 100%), đại diện cho độ sôi động, mạnh mẽ của bài hát. |
| `artist(s)_name` | Chuỗi ký tự (`str`) | Biến phân loại / Định danh | Tên (các) nghệ sĩ trình bày bài hát. |
| `released_year` | Số nguyên (`int64`) | Biến thời gian / Kiểm soát | Năm bài hát chính thức được phát hành. |

*Lưu ý về thang đo:* Hai biến độc lập chính (`danceability_%` và `energy_%`) được Spotify chuẩn hóa dưới dạng tỷ lệ phần trăm (%), giúp mô hình hồi quy OLS dễ giải thích ý nghĩa hệ số biên tế ($\beta$) hơn.

#### b. Tiền xử lý dữ liệu: Import thư viện và làm sạch dữ liệu

In [4]:
# Auto-generated code for data analysis after cleaning
%run data_cleaning.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the cleaned data
df = pd.read_csv("../datasets/processed/cleaned_spotify_2023.csv")
print(f"Dataset size for analysis: {df.shape}")

--- Downloading origin dataset ---
Initial dataset size: 953 rows, 24 columns

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    

In [5]:
selected_columns = ['streams', 'log_streams', 'danceability_%', 'energy_%', 'released_year']

print("--- Statistical Summary of Key Variables ---")
# Check data types and count of non-null values
print(df[selected_columns].info())

# Display mathematical distribution (Min, Max, Mean) for EDA preparation
display(df[selected_columns].describe().round(2))

--- Statistical Summary of Key Variables ---
<class 'pandas.DataFrame'>
RangeIndex: 952 entries, 0 to 951
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   streams         952 non-null    int64  
 1   log_streams     952 non-null    float64
 2   danceability_%  952 non-null    int64  
 3   energy_%        952 non-null    int64  
 4   released_year   952 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 37.3 KB
None


,streams,log_streams,danceability_%,energy_%,released_year
count,9.520000e+02,952.00,952.00,952.00,952.00
mean,5.141374e+08,19.51,66.98,64.27,2018.29
std,5.668569e+08,1.15,14.63,16.56,11.01
min,2.762000e+03,7.92,23.00,9.00,1930.00
25%,1.416362e+08,18.77,57.00,53.00,2020.00
50%,2.905309e+08,19.49,69.00,66.00,2022.00
75%,6.738690e+08,20.33,78.00,77.00,2022.00
max,3.703895e+09,22.03,96.00,97.00,2023.00


## 3 Phân tích khám phá dữ liệu (EDA)

### a. Phân tích đơn biến (Univariate Analysis)
* Trực quan hóa phân phối của biến mục tiêu (`streams` và `log_streams`) bằng Histogram để đánh giá độ lệch.

![Phân phối của Streams](../reports/figures/distribution_streams_plot.png)

Phân phối của streams bị lệch phải rất nặng còn phân phối của biến log_streams cho thấy dữ liệu có dạng hình chuông tương đối đối xứng, tập trung dày đặc ở khoảng giữa (từ 18 đến 20). Điều này chứng tỏ việc áp dụng phép biến đổi Logarithm đối với tổng lượt stream (streams) đã xử lý hiệu quả hiện tượng lệch phải (positive skewness) gốc của dữ liệu.
Phân phối này xấp xỉ phân phối chuẩn (Normal Distribution), đáp ứng tốt giả định quan trọng của mô hình hồi quy tuyến tính bình phương tối thiểu (OLS) sẽ được sử dụng ở các bước sau.

* Sử dụng Boxplot để xác định các bài hát "siêu hit" (Outliers vượt trội trên thị trường).

![Các bài hát siêu hit](../reports/figures/super_hits_boxplot.png)

Boxplot cho thấy phân phối lượt stream bị lệch mạnh với rất nhiều outliers nằm phía trên whisker. Điều này cho thấy thị trường streaming trên Spotify mang tính cạnh tranh rất cao, nơi chỉ một số ít bài hát đạt được mức độ phổ biến cực lớn và trở thành “super hits”. Sự tồn tại của các outliers này cho thấy cần phân tích sâu hơn các đặc tính âm nhạc và yếu tố lan tỏa để hiểu điều gì khiến một bài hát trở thành hit.

* Thống kê mô tả xu hướng trung tâm (Mean, Median) của các đặc tính âm thanh kỹ thuật số.

![Thống kê về các đặc tính âm thanh kỹ thuật số](../reports/figures/audio_features_distribution.png)

Biểu đồ Mean/Median cho thấy các đặc tính như danceability và energy có giá trị trung bình khá cao trong dataset. Điều này phản ánh xu hướng các bài hát phổ biến trên Spotify thường mang tính dễ nghe, sôi động và phù hợp với thị hiếu đại chúng. Ngoài ra, sự chênh lệch không quá lớn giữa mean và median ở nhiều đặc tính cho thấy dữ liệu tương đối ổn định và không bị ảnh hưởng quá mạnh bởi các giá trị ngoại lệ.

### b. Phân tích đặc tính âm nhạc ảnh hưởng đến độ hot (Audio Features vs. Hit Status)
* Biểu đồ Scatter plot và đường xu hướng (Trendline):
    * `danceability_%` tác động thế nào đến `log_streams`?
    * `energy_%` tác động thế nào đến `log_streams`?

![Biểu đồ scatter plot và đường xu hướng](../reports/figures/scatter_danceability_energy_logstreams_trends.png)

* Biểu đồ Boxplot phân tích nhạc lý nâng cao:
    * Mức độ thành công của bài hát dựa trên Tông nhạc (`key`) và Thể thức (`mode` - Trưởng/Thứ).
    * Tác động của tốc độ nhịp điệu (`bpm`) đến tổng lượt nghe.

![Biểu đồ scatter plot và đường xu hướng](../reports/figures/boxplot_key_mode_and_bpm.png)

* Về biểu đồ boxplot:
    Lượt nghe (log_streams) có sự phân hóa rõ rệt theo từng tông nhạc, nhưng sự khác biệt giữa hai thể thức Major (Trưởng - thường tươi sáng) và Minor (Thứ - thường trầm buồn) không quá lớn ở đa số các tông.

    Tông nhạc nổi bật: Các tông như C#, G, và B cho thấy dải hộp (IQR) nằm ở mức cao và ổn định. Đặc biệt, tông C# ở cả hai thể thức đều thu hút lượng stream lớn, cho thấy đây là tông nhạc "thời thượng" của năm 2023.

    Sự ổn định của Thể thức: Thể thức Major (màu xanh lá) có xu hướng xuất hiện nhiều Outliers (ngoại lai) cực cao hơn, ám chỉ các bài siêu hit thường có giai điệu tươi sáng. Tuy nhiên, các bài ở tông D hoặc F ở thể thức Minor lại có trung vị (median) khá cạnh tranh, cho thấy dòng nhạc tâm trạng vẫn giữ vững vị thế.

    Điểm bất thường: Có một vài bài hát ở tông C# và F (nhánh Minor) có lượt stream cực thấp (điểm đơn lẻ ở đáy biểu đồ), cho thấy không phải cứ chọn "tông hot" là sẽ thành công nếu các yếu tố khác không tốt.


* Về biểu đồ mối quan hệ giữa bpm và log streams:
    Vùng tập trung "Vàng" (Sweet Spot): Các đường đồng mức (KDE) tập trung dày đặc nhất ở khoảng 100 BPM đến 130 BPM. Đây là tốc độ nhịp điệu tiêu chuẩn của hầu hết các bài hát Pop, Dance và Hip-hop hiện đại, giúp bài hát dễ tiếp cận với đa số người nghe.

    Xu hướng phân tán:

        Các bài hát có BPM quá thấp (< 80 BPM) hoặc quá cao (> 160 BPM) có mật độ thưa thớt hơn và dải log_streams hẹp hơn. Điều này cho thấy các dòng nhạc quá chậm (như Ballad buồn) hoặc quá nhanh (như Hard-EDM) khó tiếp cận được quy mô lượt nghe tỷ view như nhạc Mid-tempo.

        Có một số ít bài hit vẫn đạt lượt stream cao ở mức 170-180 BPM, nhưng chúng đóng vai trò là các trường hợp ngoại lệ hơn là xu hướng chung của thị trường.

    Kết luận sơ bộ: Tốc độ nhịp điệu có vẻ là một yếu tố "ngưỡng" – nghĩa là bạn cần nằm trong khoảng 100-130 BPM để tối ưu hóa khả năng lọt vào các playlist phổ biến, từ đó gián tiếp thúc đẩy lượt stream.

### c. Phân tích các yếu tố lan tỏa và phân phối (Distribution & Contextual Factors)
* Biểu đồ cột so sánh: Bài hát Solo vs. Bài hát Hợp tác (`artist_count`) – Xu hướng nào dễ tạo hit hơn?

* Biểu đồ đường (Line chart): Xu hướng phát hành bài hit theo các tháng trong năm (`released_month`).

* Ma trận tương quan toàn diện (Correlation Heatmap) giữa các chỉ số âm thanh (`danceability_%`, `energy_%`, `valence_%`, `acousticness_%`) và độ phủ Playlist trên các nền tảng (Apple, Deezer, Spotify) để kiểm tra hiện tượng đa cộng tuyến.

## 4 Kiểm định giả thuyết và Mô hình hóa
### a. Phân tích tương quan (Correlation Analysis)
* **Kiểm định Pearson Correlation:** Đánh giá mức độ chặt chẽ của mối quan hệ tuyến tính giữa các audio features với biến `log_streams`.

* **Kiểm định Spearman Rank Correlation:** Kiểm tra mối quan hệ đơn điệu (phi tuyến) để phòng trường hợp dữ liệu có phân phối bất thường.

### b. Xây dựng mô hình Hồi quy tuyến tính bội (Multiple Linear Regression)
Xây dựng mô hình OLS để đánh giá đồng thời tác động của độ nhảy và năng lượng âm thanh lên quy mô lượt stream:

$$\text{log\_streams} = \beta_0 + \beta_1 \cdot \text{danceability\_\%} + \beta_2 \cdot \text{energy\_\%} + \epsilon$$

*Trong đó:* $\beta_0$ là hằng số chặn; $\beta_1, \beta_2$ là các hệ số hồi quy biên tế; $\epsilon$ là sai số ngẫu nhiên.

### c. Kiểm tra các giả định của mô hình hồi quy (OLS Assumptions Testing)
* **Giả định về phân phối chuẩn của sai số (Normality of Residuals):** Sử dụng biểu đồ Q-Q Plot và kiểm định Jarque-Bera.

* **Giả định về phương sai sai số không đổi (Homoscedasticity):** Vẽ biểu đồ Residuals vs Fitted plot để quan sát độ phân tán.

* **Kiểm tra đa cộng tuyến (Multicollinearity):** Tính toán chỉ số VIF (Variance Inflation Factor) giữa `danceability_%` và `energy_%`.

### d. Đánh giá và phân tích kết quả mô hình
* Đánh giá mức độ phù hợp của toàn bộ mô hình thông qua hệ số xác định hiệu chỉnh $R^2$ điều chỉnh ($Adjusted-R^2$).

* Kiểm định ý nghĩa của các hệ số hồi quy dựa trên giá trị trị thống kê $t$ và $p\text{-value}$ để đưa ra quyết định bác bỏ hay chấp nhận các giả thuyết $H_0/H_1$ đã đề xuất ở Mục 1.

## 5 Kết luận & Giải thích kết quả (Conclusion & Discussion)
* Trả lời trực tiếp câu hỏi nghiên cứu: Giả thuyết thuận chiều là Đúng hay Sai dựa trên dữ liệu thực tế.
* Góc nhìn thực tế: Tại sao lại có kết quả đó? (Ví dụ: Nếu không thuận chiều, có thể vì năm 2023 người nghe chuộng nhạc lofi, chillout hơn là nhạc nhảy sôi động).
* Hạn chế của đề tài: Dữ liệu chỉ gói gọn trong năm 2023, chưa tính đến yếu tố marketing, độ nổi tiếng sẵn có của nghệ sĩ (như Taylor Swift, The Weeknd).


## 6 Ý nghĩa thực tiễn (Insights)

Kết quả nghiên cứu có thể giúp:

* Nghệ sĩ tối ưu phong cách âm nhạc
* Producer hiểu xu hướng thị hiếu
* Xây dựng mô hình dự đoán độ phổ biến bài hát
* Hỗ trợ recommendation system cho nền tảng streaming